# 04 Win Rate — Logistic Regression

## Goal

For every `(character, card)` pair in `gold_card_choice_events`, restrict to occasions where the card was offered and fit

```
victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level
```

`was_picked`'s coefficient is the card's effect on win probability *holding run state at the time of the offer constant* — a cleaner signal than the raw pick/no-pick lift computed in 03, which doesn't control for anything.

This fits every card directly rather than pre-selecting a curated "cards of interest" list by raw lift and then re-estimating on the same data. That selection step is a winner's-curse setup: any card's raw lift is true effect plus sampling noise, and sorting by it preferentially keeps cards that got a lucky noise draw — that noise doesn't average out on a second look at the same rows, so the regression would end up biased toward whatever the screen happened to reward. Fitting every card sidesteps the problem entirely instead of working around it with a data split. `03`'s raw-lift screen is still worth checking against the results here as a sanity check (a card with strong raw lift but a controlled odds ratio near 1 suggests the raw lift was confounded), just not as a required input.

**Deliberately excluded:** `floor_reached` and `floors_gained` are not covariates here, even though they're in the table. Both are facts about how the run *ended*, not facts known at the time of the pick — `floor_reached` is close to deterministic of `victory` (a run that reaches floor 57 essentially won), so including it would leak the outcome into the predictors rather than control for a legitimate confounder. `floor` (the pick's own floor, i.e. `choice_floor`) is fine to include — that's "how far into the run this decision happened," known at decision time.

In [1]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("win-rate-logistic-regression")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "100")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")

Loaded e:\Projects\sts-card-choice-analysis\raw_data\gold\card_choice_events


In [ ]:
# No card-list filter — every (character, card) pair offered gets a regression, per the
# Goal section above. This collects the whole table (minus rows with nulls in these columns)
# to the driver as pandas, so expect a noticeably bigger toPandas() than a curated-list run.
regression_pd = (
    df.select("character_chosen", "card_name", "was_picked", "victory", "floor", "current_hp", "max_hp", "relic_count", "ascension_level")
    .na.drop()
    .toPandas()
)
print("Collected rows:", len(regression_pd))
print("Distinct (character, card) pairs:", regression_pd[["character_chosen", "card_name"]].drop_duplicates().shape[0])

In [5]:
regression_pd["was_picked"] = regression_pd["was_picked"].astype(int)
regression_pd["victory"] = regression_pd["victory"].astype(int)
regression_pd = regression_pd[regression_pd["max_hp"] > 0].copy()
regression_pd["hp_ratio"] = regression_pd["current_hp"] / regression_pd["max_hp"]

MIN_REGRESSION_ROWS = 200

# groupby().apply() needs every group to return a Series with the same keys — a group
# returning fewer keys than another (e.g. just n/error on failure) makes pandas fall back
# to a stacked long-format result instead of one row per group, so every branch below
# fills the full set of keys even when most are None.
EMPTY_RESULT = {
    "n": None,
    "error": None,
    "was_picked_coef": None,
    "was_picked_pvalue": None,
    "odds_ratio": None,
    "odds_ratio_ci_low": None,
    "odds_ratio_ci_high": None,
}

def fit_card_logit(group):
    result = dict(EMPTY_RESULT)
    if len(group) < MIN_REGRESSION_ROWS:
        result["n"] = len(group)
        result["error"] = "too few rows"
        return pd.Series(result)
    try:
        model = smf.logit(
            "victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level",
            data=group,
        ).fit(disp=0)
    except Exception as exc:
        result["n"] = len(group)
        result["error"] = str(exc)
        return pd.Series(result)
    coef = model.params["was_picked"]
    ci_low, ci_high = model.conf_int().loc["was_picked"]
    result.update({
        "n": len(group),
        "error": None,
        "was_picked_coef": coef,
        "was_picked_pvalue": model.pvalues["was_picked"],
        "odds_ratio": np.exp(coef),
        "odds_ratio_ci_low": np.exp(ci_low),
        "odds_ratio_ci_high": np.exp(ci_high),
    })
    return pd.Series(result)

results_pd = (
    regression_pd.groupby(["character_chosen", "card_name"])
    .apply(fit_card_logit, include_groups=False)
    .reset_index()
)
print("Fitted:", (results_pd["error"].isna()).sum(), "of", len(results_pd))

e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\stats

Fitted: 156 of 160


### Results

`odds_ratio` > 1 means picking the card is associated with higher win odds after controlling for HP ratio, floor, relic count, and ascension at the time it was offered; < 1 means lower. Sorted within each character by odds ratio, restricted to statistically significant results (`was_picked_pvalue < 0.05`) — cards that didn't reach significance at this sample size are dropped from this view but still available in `results_pd`.

In [6]:
significant = results_pd[
    results_pd["error"].isna() & (results_pd["was_picked_pvalue"] < 0.05)
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])

cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue"]
significant[cols]

,character_chosen,card_name,n,odds_ratio,odds_ratio_ci_low,odds_ratio_ci_high,was_picked_pvalue
23,DEFECT,Master of Strategy+1,1206.0,1.947548,1.506740,2.517317,3.562503e-07
17,DEFECT,Glacier,431375.0,1.604859,1.576619,1.633605,0.000000e+00
7,DEFECT,Coolheaded+1,265829.0,1.470030,1.442337,1.498255,0.000000e+00
1,DEFECT,Biased Cognition,326735.0,1.454079,1.426963,1.481710,0.000000e+00
22,DEFECT,Master of Strategy,13369.0,1.449472,1.308492,1.605641,1.159284e-12
...,...,...,...,...,...,...,...
155,WATCHER,Violence,9159.0,1.156412,1.022826,1.307444,2.032375e-02
142,WATCHER,Sanctity,310276.0,1.146835,1.118565,1.175819,5.397436e-27
150,WATCHER,TalkToTheHand+1,65334.0,1.134306,1.092799,1.177390,3.457154e-11
140,WATCHER,Ragnarok,232379.0,1.090169,1.066459,1.114407,1.415251e-14


## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [7]:
spark.stop()